# Switch-Centered Local PH: Stats and Figures

This notebook models local topology as a function of time from regime switches and stratifies switch effects by transition type, regime stability, posterior jump magnitude, and clinical valence direction.

Core model:
$$
\text{Topo}_{it} \sim f(\text{time from switch}) + (1\mid\text{session})
$$

Primary topology outcomes:
- `total_persistence_h1`
- `betti_h1_auc`
- `persistence_entropy_h1`

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from patsy import dmatrix
from scipy.stats import spearmanr, wilcoxon
import statsmodels.api as sm
from statsmodels.regression.mixed_linear_model import MixedLM

warnings.filterwarnings('ignore')

plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.grid'] = False
plt.rcParams['svg.fonttype'] = 'none'

ARTIFACTS_DIR = Path('../artifacts')
CHMM_DIR = ARTIFACTS_DIR / 'chmm_8_full_dataset'
PHASE_DIR = ARTIFACTS_DIR / 'regime_geometry' / 'phase_transitions'

FIG_DIR = PHASE_DIR / 'notebook_figures'
TAB_DIR = PHASE_DIR / 'notebook_tables'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

print('CHMM_DIR exists:', CHMM_DIR.exists())
print('PHASE_DIR exists:', PHASE_DIR.exists())

## 1. Load Phase-Transition Topology Outputs

If files are missing, run: `make phase-transitions` and re-run this notebook.

In [ ]:
rolling_path = PHASE_DIR / 'rolling_ph_timeseries.csv'
aligned_path = PHASE_DIR / 'switch_aligned_rolling_ph.csv'
combined_aligned_path = PHASE_DIR / 'combined_aligned_rolling_ph.csv'
switch_path = PHASE_DIR / 'switch_local_segment_ph.csv'
delta_path = PHASE_DIR / 'switch_reorganization_delta.csv'
jump_assoc_path = PHASE_DIR / 'switch_jump_association.csv'
transition_summary_path = PHASE_DIR / 'switch_transition_type_reorganization.csv'
matched_anchors_path = PHASE_DIR / 'matched_real_pseudo_anchors.csv'

required = [combined_aligned_path, switch_path]
missing = [p for p in required if not p.exists()]

if missing:
    print('Missing core phase-transition outputs:')
    for p in missing:
        print(' -', p)
    print('Run `make phase-transitions` first.')
    df_aligned = pd.DataFrame()
    df_switch = pd.DataFrame()
    df_rolling = pd.DataFrame()
    df_matched_anchors = pd.DataFrame()
else:
    df_aligned = pd.read_csv(combined_aligned_path)
    df_switch = pd.read_csv(switch_path)
    df_rolling = pd.read_csv(rolling_path) if rolling_path.exists() else pd.DataFrame()
    df_matched_anchors = pd.read_csv(matched_anchors_path) if matched_anchors_path.exists() else pd.DataFrame()
    print('Loaded aligned rows (real+pseudo):', df_aligned.shape)
    print('Loaded switch rows:', df_switch.shape)
    print('Loaded rolling rows:', df_rolling.shape)
    print('Loaded matched anchor rows:', df_matched_anchors.shape)

display(df_aligned.head(3) if not df_aligned.empty else pd.DataFrame())

## 2. Build Switch-Type Stratification Labels

We add the requested switch partitions:
1. Transition identity (`from_state -> to_state`)
2. Regime stability (`out_of_persistent`, `into_persistent`, etc.)
3. Posterior jump magnitude (`small`, `medium`, `large`)
4. Clinical valence direction (toward more negative vs neutral/stabilizing)

In [ ]:
if df_aligned.empty or df_switch.empty:
    print('No aligned/switch data available yet.')
    df_switch_labels = pd.DataFrame()
else:
    # Base switch-level table for real switches
    key_cols = ['session_id', 'switch_index', 'from_state', 'to_state', 'transition_type', 'posterior_jump_l2']
    keep_cols = [c for c in key_cols if c in df_switch.columns]
    df_switch_labels = df_switch[keep_cols].drop_duplicates(['session_id', 'switch_index']).copy()

    # 1) Transition identity
    df_switch_labels['transition_identity'] = df_switch_labels['transition_type'].astype(str)

    # 3) Posterior jump magnitude (small/medium/large)
    if 'posterior_jump_l2' in df_switch_labels.columns and df_switch_labels['posterior_jump_l2'].notna().sum() >= 6:
        q1, q2 = df_switch_labels['posterior_jump_l2'].quantile([1/3, 2/3])
        def jump_bin(x):
            if pd.isna(x):
                return 'unknown'
            if x <= q1:
                return 'small'
            if x <= q2:
                return 'medium'
            return 'large'
        df_switch_labels['jump_magnitude_bin'] = df_switch_labels['posterior_jump_l2'].map(jump_bin)
    else:
        df_switch_labels['jump_magnitude_bin'] = 'unknown'

    # 2) Regime stability: detect whether switch leaves/enters persistent runs.
    dec = pd.read_csv(CHMM_DIR / 'decoded_sessions.csv')
    path_map = {}
    for _, r in dec.iterrows():
        pf = CHMM_DIR / r['path_file']
        if pf.exists():
            path_map[str(r['session_id'])] = np.load(pf).astype(int)

    run_info_rows = []
    for sid, path in path_map.items():
        if len(path) < 2:
            continue
        run_start = 0
        switch_counter = 0
        for t in range(1, len(path)):
            if path[t] != path[t - 1]:
                from_run_len = t - run_start
                # next run length
                run_end = t + 1
                while run_end < len(path) and path[run_end] == path[t]:
                    run_end += 1
                to_run_len = run_end - t

                run_info_rows.append({
                    'session_id': sid,
                    'switch_index': switch_counter,
                    'from_run_len': int(from_run_len),
                    'to_run_len': int(to_run_len),
                })
                switch_counter += 1
                run_start = t

    df_runs = pd.DataFrame(run_info_rows)
    if not df_runs.empty:
        persistent_thresh = float(df_runs[['from_run_len', 'to_run_len']].stack().quantile(0.75))
        df_runs['from_persistent'] = df_runs['from_run_len'] >= persistent_thresh
        df_runs['to_persistent'] = df_runs['to_run_len'] >= persistent_thresh

        def stability_type(r):
            if r['from_persistent'] and (not r['to_persistent']):
                return 'out_of_persistent'
            if (not r['from_persistent']) and r['to_persistent']:
                return 'into_persistent'
            if r['from_persistent'] and r['to_persistent']:
                return 'persistent_to_persistent'
            return 'nonpersistent_to_nonpersistent'

        df_runs['regime_stability_type'] = df_runs.apply(stability_type, axis=1)
        df_switch_labels = df_switch_labels.merge(
            df_runs[['session_id', 'switch_index', 'regime_stability_type', 'from_run_len', 'to_run_len']],
            on=['session_id', 'switch_index'],
            how='left'
        )
    else:
        df_switch_labels['regime_stability_type'] = 'unknown'

    # 4) Clinical valence direction from client-emission profile in best_model.json.
    # Assumes client code ordering is [negative, neutral, positive].
    model_json = CHMM_DIR / 'best_model.json'
    if model_json.exists():
        params = json.loads(model_json.read_text())
        pC = np.asarray(params['pC'], dtype=float)  # shape: (K,3)
        valence_score = pC @ np.array([-1.0, 0.0, 1.0])

        def valence_direction(fr, to):
            if pd.isna(fr) or pd.isna(to) or fr < 0 or to < 0: # Handle pseudo-switches
                return 'unknown'
            fr = int(fr); to = int(to)
            if fr >= len(valence_score) or to >= len(valence_score):
                return 'unknown'
            dv = float(valence_score[to] - valence_score[fr])
            if dv <= -0.1:
                return 'toward_more_negative_client_affect'
            if dv >= 0.1:
                return 'toward_neutral_or_stabilizing'
            return 'minimal_or_mixed_direction'

        df_switch_labels['clinical_valence_direction'] = df_switch_labels.apply(
            lambda r: valence_direction(r.get('from_state', np.nan), r.get('to_state', np.nan)), axis=1
        )
    else:
        df_switch_labels['clinical_valence_direction'] = 'unknown'

    # Join labels onto aligned rows
    # For pseudo-switches, these labels will be mostly NaN, which is expected.
    df_aligned = df_aligned.merge(
        df_switch_labels[
            [
                'session_id', 'switch_index', 'transition_identity', 'jump_magnitude_bin',
                'regime_stability_type', 'clinical_valence_direction'
            ]
        ],
        on=['session_id', 'switch_index'],
        how='left'
    )
    
    # Fill NaN values for pseudo-switches for cleaner grouping
    for col in ['transition_identity', 'jump_magnitude_bin', 'regime_stability_type', 'clinical_valence_direction']:
        if col in df_aligned.columns:
            df_aligned[col] = df_aligned[col].fillna('pseudo')


    print('Labeled switch rows:', df_switch_labels.shape)
    print('Aligned rows after label join:', df_aligned.shape)
    display(df_switch_labels.head(5))
    display(df_aligned.query("anchor_kind == 'pseudo'").head(5))

## 3. Mixed-Effects Time Model

Model each topology outcome as a smooth function of time from switch with random intercept by session:
$$
\text{Topo}_{it} \sim f(\text{rel\_time}_{it}) + (1\mid\text{session})
$$

Here `f` is represented using cubic spline basis.

In [ ]:
TOPO_FEATURES = ['total_persistence_h1', 'betti_h1_auc', 'persistence_entropy_h1']

def _full_rank_design(X, tol=1e-10):
    """Drop near-constant non-const columns and trim to full rank."""
    X = X.copy()

    # Keep explicit intercept if present.
    has_const = 'const' in X.columns
    const_col = X[['const']].copy() if has_const else None

    non_const_cols = [c for c in X.columns if c != 'const']
    if len(non_const_cols) > 0:
        Xn = X[non_const_cols]
        keep = Xn.columns[Xn.std(axis=0) > tol]
        Xn = Xn[keep]

        # Iteratively trim non-const columns until full rank.
        while Xn.shape[1] > 0 and np.linalg.matrix_rank(Xn.to_numpy()) < Xn.shape[1]:
            Xn = Xn.iloc[:, :-1]
    else:
        Xn = pd.DataFrame(index=X.index)

    if has_const:
        Xout = pd.concat([const_col, Xn], axis=1)
    else:
        Xout = Xn

    # Final safety: if rank still deficient with const included, trim non-const further.
    while Xout.shape[1] > 1 and np.linalg.matrix_rank(Xout.to_numpy()) < Xout.shape[1]:
        if 'const' in Xout.columns and Xout.columns[-1] == 'const':
            break
        Xout = Xout.iloc[:, :-1]

    return Xout

def fit_mixed_spline(df, feature, spline_df_candidates=(5, 4, 3)):
    d = df[['session_id', 'rel_time_sec', feature]].dropna().copy()
    if len(d) < 40 or d['session_id'].nunique() < 4:
        return None, None

    from patsy import build_design_matrices

    last_err = None

    # Try progressively simpler spline bases to avoid singularity.
    for sdf in spline_df_candidates:
        formula = f'bs(rel_time_sec, df={sdf}, degree=3, include_intercept=False)'
        X_basis = dmatrix(formula, d, return_type='dataframe')
        design_info = X_basis.design_info
        X = sm.add_constant(X_basis, has_constant='add')
        X = _full_rank_design(X)
        if 'const' not in X.columns:
            X = sm.add_constant(X, has_constant='add')
        if X.shape[1] < 2:
            continue

        # Primary model: mixed effects with session random intercept.
        try:
            model = MixedLM(endog=d[feature], exog=X, groups=d['session_id'])
            fit = model.fit(reml=False, method='lbfgs', maxiter=300)
            return d, {
                'fit': fit,
                'model_type': 'MixedLM',
                'spline_df': int(sdf),
                'X_cols': list(X.columns),
                'design_info': design_info,
            }
        except Exception as e:
            last_err = e

        # Fallback: fixed-effects spline OLS (no random effect), keeps analysis running.
        try:
            fit = sm.OLS(d[feature], X).fit()
            print(f'Fallback to OLS spline for {feature} (spline_df={sdf}) due to MixedLM failure.')
            return d, {
                'fit': fit,
                'model_type': 'OLS_spline_fallback',
                'spline_df': int(sdf),
                'X_cols': list(X.columns),
                'design_info': design_info,
            }
        except Exception as e2:
            last_err = e2

    print(f'Skipping {feature}: unable to fit model ({type(last_err).__name__}: {last_err})')
    return None, None

def predict_curve(fit_bundle, t_min, t_max, n=200):
    from patsy import build_design_matrices

    fit = fit_bundle['fit']
    x_cols = fit_bundle['X_cols']
    design_info = fit_bundle['design_info']

    grid = pd.DataFrame({'rel_time_sec': np.linspace(t_min, t_max, n)})
    Xg_basis = build_design_matrices([design_info], grid, return_type='dataframe')[0]
    Xg = sm.add_constant(Xg_basis, has_constant='add')

    # Align prediction design to training columns.
    for c in x_cols:
        if c not in Xg.columns:
            Xg[c] = 0.0
    Xg = Xg[x_cols]

    if 'const' not in Xg.columns:
        Xg = sm.add_constant(Xg, has_constant='add')

    grid['pred'] = fit.predict(exog=Xg)
    return grid

if df_aligned.empty:
    print('No aligned data available yet.')
    df_time_model = pd.DataFrame()
else:
    model_rows = []
    model_fits = {}
    for f in TOPO_FEATURES:
        d, fit_bundle = fit_mixed_spline(df_aligned, f, spline_df_candidates=(5, 4, 3))
        if fit_bundle is None:
            continue
        model_fits[f] = (d, fit_bundle)
        fit = fit_bundle['fit']
        model_rows.append({
            'feature': f,
            'model_type': fit_bundle['model_type'],
            'spline_df_used': fit_bundle['spline_df'],
            'n_obs': int(len(d)),
            'n_sessions': int(d['session_id'].nunique()),
            'aic': float(getattr(fit, 'aic', np.nan)),
            'bic': float(getattr(fit, 'bic', np.nan)),
            'llf': float(getattr(fit, 'llf', np.nan)),
        })

    df_time_model = pd.DataFrame(model_rows)
    display(df_time_model)
    if not df_time_model.empty:
        df_time_model.to_csv(TAB_DIR / 'mixed_time_model_summary.csv', index=False)

        # Plot fitted time-course per feature.
        n_feat = len(model_fits)
        fig, axes = plt.subplots(n_feat, 1, figsize=(10, max(4, 3.5 * n_feat)), sharex=True)
        if n_feat == 1:
            axes = [axes]

        for ax, f in zip(axes, model_fits.keys()):
            d, fit_bundle = model_fits[f]
            tmin, tmax = d['rel_time_sec'].min(), d['rel_time_sec'].max()
            pred = predict_curve(fit_bundle, tmin, tmax, n=200)

            avg = d.groupby(pd.cut(d['rel_time_sec'], bins=40, duplicates='drop'))[f].mean().reset_index()
            avg['rel_mid'] = avg['rel_time_sec'].apply(lambda x: x.mid if pd.notna(x) else np.nan)

            ax.plot(pred['rel_time_sec'], pred['pred'], color='black', linewidth=2, label='Model fit')
            ax.scatter(avg['rel_mid'], avg[f], s=16, alpha=0.5, color='tab:blue', label='Binned means')
            ax.axvline(0.0, color='red', linestyle='--', linewidth=1.0)
            ax.set_title(f"{f} ~ f(time from switch), {fit_bundle['model_type']}")
            ax.set_ylabel(f)
            ax.legend(loc='best')

        axes[-1].set_xlabel('Time relative to switch (sec)')
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'mixed_time_model_fitted_curves.svg', format='svg')
        plt.show()

## 4. Switch Splits: Time-Course by Type

We summarize switch-centered trajectories by each requested grouping and plot mean curves for the three topology outcomes.

In [ ]:
def plot_split_timecourses(df, split_col, features, out_prefix, min_group_n=30):
    d = df.copy()
    if split_col not in d.columns:
        print(f'Missing split column: {split_col}')
        return pd.DataFrame()

    # Keep reasonably represented groups.
    counts = d.groupby('session_id')[split_col].nunique()
    valid_sessions = counts[counts > 1].index if split_col == 'anchor_kind' else d['session_id'].unique()
    
    d = d[d['session_id'].isin(valid_sessions)].copy()

    group_counts = d.groupby(split_col)['session_id'].nunique()
    keep = group_counts[group_counts >= min_group_n].index
    d = d[d[split_col].isin(keep)].copy()

    if d.empty:
        print(f'No groups with >= {min_group_n} sessions for {split_col}')
        return pd.DataFrame()

    summary_rows = []
    for f in features:
        grp = d.groupby([split_col, 'rel_time_sec'], as_index=False)[f].mean()
        grp['feature'] = f
        grp = grp.rename(columns={f: 'value'})
        summary_rows.append(grp)
    df_sum = pd.concat(summary_rows, ignore_index=True)

    n_feat = len(features)
    fig, axes = plt.subplots(n_feat, 1, figsize=(11, max(4, 3.4 * n_feat)), sharex=True)
    if n_feat == 1:
        axes = [axes]

    for ax, f in zip(axes, features):
        sub = df_sum[df_sum['feature'] == f]
        sns.lineplot(data=sub, x='rel_time_sec', y='value', hue=split_col, linewidth=2, ax=ax)
        ax.axvline(0.0, color='red', linestyle='--', linewidth=1.0)
        ax.set_title(f'{f} by {split_col}')
        ax.set_ylabel(f)
        ax.legend(loc='best', fontsize=8)

    axes[-1].set_xlabel('Time relative to switch (sec)')
    plt.tight_layout()
    fig.savefig(FIG_DIR / f'{out_prefix}_timecourses.svg', format='svg')
    plt.show()

    df_sum.to_csv(TAB_DIR / f'{out_prefix}_timecourses.csv', index=False)
    return df_sum

if df_aligned.empty:
    print('No aligned data available yet.')
else:
    _ = plot_split_timecourses(df_aligned, 'anchor_kind', TOPO_FEATURES, 'split_anchor_kind', min_group_n=10)
    _ = plot_split_timecourses(df_aligned, 'transition_identity', TOPO_FEATURES, 'split_transition_identity', min_group_n=50)
    _ = plot_split_timecourses(df_aligned, 'regime_stability_type', TOPO_FEATURES, 'split_regime_stability', min_group_n=40)
    _ = plot_split_timecourses(df_aligned, 'jump_magnitude_bin', TOPO_FEATURES, 'split_jump_magnitude', min_group_n=40)
    _ = plot_split_timecourses(df_aligned, 'clinical_valence_direction', TOPO_FEATURES, 'split_valence_direction', min_group_n=40)

## 5. Before vs During vs After Switch

Test whether topology changes across pre/during/post phases and whether changes scale with posterior jump magnitude.

In [ ]:
if df_aligned.empty:
    print('No aligned data available yet.')
else:
    phase_summary_rows = []
    model_results_rows = []
    delta_model_results_rows = []

    for f in TOPO_FEATURES:
        d = df_aligned[['session_id', 'switch_index', 'anchor_kind', 'phase', 'posterior_jump_l2', f]].dropna(subset=[f, 'phase']).copy()
        if d.empty:
            continue

        # Per-switch phase mean for paired contrasts, stratified by anchor_kind
        sw = d.groupby(['session_id', 'switch_index', 'anchor_kind', 'phase'], as_index=False)[f].mean()
        wide = sw.pivot_table(index=['session_id', 'switch_index', 'anchor_kind'], columns='phase', values=f).reset_index()
        
        # Merge jumps for real anchors only
        jumps = d[d['anchor_kind'] == 'real'][['session_id', 'switch_index', 'posterior_jump_l2']].drop_duplicates()
        wide = wide.merge(jumps, on=['session_id', 'switch_index'], how='left')

        for kind in ['real', 'pseudo']:
            kind_wide = wide[wide['anchor_kind'] == kind]
            for contrast_name, c1, c2 in [
                ('during_minus_before', 'before', 'during'),
                ('after_minus_before', 'before', 'after'),
                ('after_minus_during', 'during', 'after'),
            ]:
                if c1 in kind_wide.columns and c2 in kind_wide.columns:
                    delta_df = kind_wide[[c1, c2, 'session_id']].dropna()
                    delta = delta_df[c2] - delta_df[c1]
                    if len(delta) < 8:
                        continue
                    
                    w_stat, w_p = wilcoxon(delta)
                    
                    md = MixedLM.from_formula("delta ~ 1", data=pd.DataFrame({'delta': delta}), groups=delta_df['session_id']).fit(reml=False)
                    
                    phase_summary_rows.append({
                        'feature': f,
                        'anchor_kind': kind,
                        'contrast': contrast_name,
                        'n_switches': int(len(delta)),
                        'mean_delta': float(np.mean(delta)),
                        'median_delta': float(np.median(delta)),
                        'wilcoxon_stat': w_stat,
                        'wilcoxon_p': w_p,
                        'mixed_lm_t': md.tvalues[0],
                        'mixed_lm_p': md.pvalues[0],
                    })
        
        # Models on deltas
        for contrast_name, c1, c2 in [
            ('after_minus_before', 'before', 'after'),
            ('after_minus_during', 'during', 'after'),
        ]:
            if c1 in wide.columns and c2 in wide.columns:
                delta_data = wide[['session_id', 'anchor_kind', c1, c2]].dropna()
                delta_data['delta'] = delta_data[c2] - delta_data[c1]
                
                if delta_data['session_id'].nunique() > 1 and len(delta_data) > 10:
                    try:
                        formula = f"delta ~ C(anchor_kind, Treatment('pseudo'))"
                        model = MixedLM.from_formula(formula, data=delta_data, groups='session_id').fit(reml=False)
                        
                        summary_table = model.summary().tables[1]
                        if hasattr(summary_table, 'data'):
                            res = pd.DataFrame(summary_table.data[1:], columns=summary_table.data[0])
                        else:
                            res = summary_table.iloc[1:]
                            
                        res['feature'] = f
                        res['contrast'] = contrast_name
                        delta_model_results_rows.append(res)
                    except Exception as e:
                        print(f"Could not fit delta model for {f}, contrast {contrast_name}: {e}")


        # Jump association with after-before reorganization (only for real switches)
        real_wide = wide[wide['anchor_kind'] == 'real']
        if 'before' in real_wide.columns and 'after' in real_wide.columns:
            dd = real_wide[['posterior_jump_l2', 'before', 'after']].dropna().copy()
            if len(dd) >= 8:
                dd['delta_after_minus_before'] = dd['after'] - dd['before']
                r, p = spearmanr(dd['posterior_jump_l2'], dd['delta_after_minus_before'])
                phase_summary_rows.append({
                    'feature': f,
                    'anchor_kind': 'real',
                    'contrast': 'jump_vs_after_before_delta',
                    'n_switches': int(len(dd)),
                    'mean_delta': np.nan,
                    'median_delta': np.nan,
                    'wilcoxon_stat': float(r),
                    'wilcoxon_p': float(p),
                    'mixed_lm_t': np.nan,
                    'mixed_lm_p': np.nan,
                })

        # Figure: phase boxplots by anchor_kind
        plt.figure(figsize=(8, 5))
        sns.boxplot(data=d, x='phase', y=f, hue='anchor_kind', order=['before', 'during', 'after'])
        plt.title(f'{f}: before vs during vs after by anchor type')
        plt.tight_layout()
        plt.savefig(FIG_DIR / f'phase_boxplot_{f}_by_anchor.svg', format='svg')
        plt.show()

        # Mixed model: value ~ phase * anchor_kind + (1 | session_id)
        model_data = d[['session_id', 'phase', 'anchor_kind', f]].dropna()
        model_data['phase'] = pd.Categorical(model_data['phase'], categories=['before', 'during', 'after'], ordered=True)
        
        if model_data['session_id'].nunique() > 1:
            try:
                formula = f"{f} ~ C(phase, Treatment('before')) * C(anchor_kind, Treatment('pseudo'))"
                model = MixedLM.from_formula(formula, data=model_data, groups='session_id').fit(reml=False)
                
                summary_table = model.summary().tables[1]
                if hasattr(summary_table, 'data'):
                    res = pd.DataFrame(summary_table.data[1:], columns=summary_table.data[0])
                else:
                    res = summary_table.iloc[1:]

                res['feature'] = f
                model_results_rows.append(res)
            except Exception as e:
                print(f"Could not fit mixed model for {f}: {e}")


    df_phase_stats = pd.DataFrame(phase_summary_rows) if phase_summary_rows else pd.DataFrame()
    df_model_results = pd.concat(model_results_rows, ignore_index=True) if model_results_rows else pd.DataFrame()
    df_delta_model_results = pd.concat(delta_model_results_rows, ignore_index=True) if delta_model_results_rows else pd.DataFrame()


    display(df_phase_stats)
    if not df_phase_stats.empty:
        df_phase_stats.to_csv(TAB_DIR / 'phase_contrast_and_jump_stats.csv', index=False)
        print('Saved phase contrast/jump summary table.')

    display(df_model_results)
    if not df_model_results.empty:
        df_model_results.to_csv(TAB_DIR / 'phase_anchor_kind_mixed_model.csv', index=False)
        print('Saved phase x anchor_kind mixed model results.')

    display(df_delta_model_results)
    if not df_delta_model_results.empty:
        df_delta_model_results.to_csv(TAB_DIR / 'delta_anchor_kind_mixed_model.csv', index=False)
        print('Saved delta x anchor_kind mixed model results.')

## 6. Transition-Type Reorganization Focus

For interpretability, this section ranks transition identities by mean topology reorganization (after minus before).

In [ ]:
if df_aligned.empty:
    print('No aligned data available yet.')
else:
    rows = []
    top_identity_keep = None

    for f in TOPO_FEATURES:
        d = df_aligned[['session_id', 'switch_index', 'transition_identity', 'phase', f]].dropna().copy()
        if d.empty:
            continue

        sw = d.groupby(['session_id', 'switch_index', 'transition_identity', 'phase'], as_index=False)[f].mean()
        wide = sw.pivot_table(index=['session_id', 'switch_index', 'transition_identity'], columns='phase', values=f).reset_index()
        if 'before' not in wide.columns or 'after' not in wide.columns:
            continue

        wide['delta_after_minus_before'] = wide['after'] - wide['before']
        agg = (
            wide.groupby('transition_identity')['delta_after_minus_before']
            .agg(mean_delta='mean', median_delta='median', n_switches='count')
            .reset_index()
        )
        agg['feature'] = f

        if top_identity_keep is None:
            top_identity_keep = (
                agg.sort_values('n_switches', ascending=False)
                .head(10)['transition_identity']
                .tolist()
            )
        rows.append(agg)

        plot_df = agg[agg['transition_identity'].isin(top_identity_keep)].sort_values('mean_delta', ascending=False)
        if plot_df.empty:
            continue

        plt.figure(figsize=(9, 5))
        sns.barplot(data=plot_df, x='mean_delta', y='transition_identity', color='steelblue')
        plt.axvline(0.0, color='black', linestyle='--', linewidth=1.0)
        plt.title(f'Transition identity reorganization ranking ({f})')
        plt.xlabel('Mean after-before delta')
        plt.ylabel('Transition identity')
        plt.tight_layout()
        plt.savefig(FIG_DIR / f'transition_identity_delta_rank_{f}.svg', format='svg')
        plt.show()

    df_transition_rank = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    display(df_transition_rank.head(20) if not df_transition_rank.empty else pd.DataFrame())
    if not df_transition_rank.empty:
        df_transition_rank.to_csv(TAB_DIR / 'transition_identity_reorganization_rankings.csv', index=False)